# Minor Project: methodology runs (Colab)

Early crop type classification on CatCrops (Lleida). Runs the ablation ladder
A (baseline) -> B (dual-branch + FiLM/cross-attention) -> C (+ rotation gate) -> D (+ CB focal loss).

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Everything that must survive a
disconnect (processed data, checkpoints, predictions) is saved to Google Drive, so after a
disconnect just run all cells again: finished work is skipped and training resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch the runtime to T4 GPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/crop-methodology'
CACHE = f'{DRIVE}/cache'      # processed data (made once)
RUNS  = f'{DRIVE}/runs'       # checkpoints, logs, predictions
import os; os.makedirs(CACHE, exist_ok=True); os.makedirs(RUNS, exist_ok=True)

## Get the code
The repository is private. Add a GitHub token with read access to it under
**Secrets** (key icon in the left bar) with the name `GH_TOKEN`, and set `REPO` below.

In [ ]:
REPO = 'CHANGE-ME/crop-methodology'   # GitHub user/repo
import os, subprocess
from google.colab import userdata
token = userdata.get('GH_TOKEN')
code_dir = '/content/crop-methodology'
if not os.path.exists(code_dir):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@github.com/{REPO}.git', code_dir],
                       capture_output=True, text=True)
    print('cloned' if r.returncode == 0 else r.stderr.replace(token, '***'))
else:
    print(subprocess.run(['git', '-C', code_dir, 'pull', '-q'], capture_output=True, text=True).stdout or 'up to date')
%cd /content/crop-methodology
!git log --oneline -1

In [ ]:
!pip install -q kagglehub pytest

## 1. Tests (must all pass before any training)

In [ ]:
!python -m pytest -q tests

## 2. Data: download CatCrops and build the cache (only the first time)

In [ ]:
import os
if not (os.path.exists(f'{CACHE}/2022.npz') and os.path.exists(f'{CACHE}/2023.npz')):
    import kagglehub
    path = kagglehub.dataset_download('irtaremotesensing/catcrops-dataset')
    !python -m data.preprocess --root "{path}" --out "{CACHE}"
else:
    print('cache already built')
!head -40 "{CACHE}/2022.json"

## 3. Smoke run on real data (2,000 fields, 1 epoch) to time an epoch

In [ ]:
!python train.py --config configs/D_full.yaml --cache "{CACHE}" --out /content/smoke --limit 2000 --epochs 1 --force

## 4. Ablation ladder A -> D, seeds 0, 1, 2
Stops itself before the session budget runs out; rerun this cell in a new session to continue.

In [ ]:
!python run_experiments.py --cache "{CACHE}" --out "{RUNS}" --seeds 0 1 2 --max-minutes 200

## 5. Results so far (2023 test year, mean over seeds)

In [ ]:
import pandas as pd
p = f'{RUNS}/results_summary.csv'
if os.path.exists(p):
    t = pd.read_csv(p)
    t['_d'] = t['date'].map({'01Mar': 0, '01Jun': 1, '01Sep': 2, '01Dec': 3})
    t = t.sort_values(['run', '_d']).drop(columns='_d')
    display(t[['run', 'date', 'accuracy_mean', 'accuracy_std', 'macro_f1_mean', 'macro_f1_std',
               'changed_macro_f1_mean', 'accuracy_count']].round(4))
else:
    print('no finished runs yet')

## Optional extras (switchable fixes, off in runs A-D)

In [ ]:
# !python run_experiments.py --cache "{CACHE}" --out "{RUNS}" --configs D_full_prior_dropout D_full_cloud_weighted --seeds 0 --max-minutes 200